In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer


In [5]:
misconception = pd.read_csv('data/misconception_mapping.csv')
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [30]:
import pandas as pd
import torch
from transformers import AutoTokenizer

class DatasetCreator:
    def __init__(self, tokenizer_name, max_length=512):
        """
        Initializes the DatasetCreator class.
        
        Parameters:
        - tokenizer_name: The name of the tokenizer (e.g., "gpt2", "bert-base-uncased")
        - max_length: Maximum length of the input sequence
        """
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length

    def create_input_text(self, question, correct_answer, incorrect_answer, construct_name=None, subject_name=None, special_formatting=False):
        """
        Creates input text based on the question, correct answer, incorrect answer, construct name, and subject name.
        
        Parameters:
        - question: The question text
        - correct_answer: The correct answer text
        - incorrect_answer: The incorrect answer text
        - construct_name: The name of the construct (optional)
        - subject_name: The name of the subject (optional)
        - special_formatting: Whether to use special formatting for the prompt (optional)
        
        Returns:
        - input_text: Formatted text ready for tokenization
        """
        if special_formatting:
            input_text = (
                f"Here is a question for you:\n"
                f"{question}\n"
                f"The correct answer is: {correct_answer}\n"
                f"However, a common misconception is: {incorrect_answer}\n"
            )
            if construct_name:
                input_text += f"This question is related to: {construct_name}.\n"
            if subject_name:
                input_text += f"Subject area: {subject_name}."
        else:
            messages = [
                f"Question: {question}",
                f"Correct Answer: {correct_answer}",
                f"Incorrect Answer: {incorrect_answer}"
            ]
            
            if construct_name:
                messages.append(f"Construct Name: {construct_name}")
            if subject_name:
                messages.append(f"Subject Name: {subject_name}")
            
            input_text = "\n".join(messages)
        
        return input_text
    
    def tokenize_data(self, input_text):
        """
        Tokenizes the input text using the provided tokenizer.
        
        Parameters:
        - input_text: The input text to be tokenized
        
        Returns:
        - tokenized_data: Tokenized representation of the input text
        """
        return self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
    
    def create_dataset(self, df, misconception_mapping, include_construct_name=False, include_subject_name=False, special_formatting=False, testing=False):
        """
        Creates a dataset for direct preference optimization (DPO).
        
        Parameters:
        - df: The training DataFrame containing all the necessary columns
        - misconception_mapping: A DataFrame containing the Misconception IDs and their names
        - include_construct_name: Whether to include the construct name in the prompt (optional)
        - include_subject_name: Whether to include the subject name in the prompt (optional)
        - special_formatting: Whether to use special formatting for the prompts (optional)
        
        Returns:
        - A dictionary containing tokenized input tensors for the DPO model
        """
        inputs, labels = [], []
        misconception_dict = pd.Series(misconception_mapping.MisconceptionName.values, index=misconception_mapping.MisconceptionId).to_dict()

        if testing:
            df = df.head(3)
        
        for index, row in df.iterrows():
            question = row['QuestionText']
            correct_answer = row[f'Answer{row.CorrectAnswer}Text']
            
            for answer_label in ['A', 'B', 'C', 'D']:
                misconception_id = row.get(f'Misconception{answer_label}Id', None)
                if pd.notna(misconception_id):
                    incorrect_answer = row[f'Answer{answer_label}Text']
                    misconception_name = misconception_dict.get(misconception_id, "Unknown Misconception")
                    
                    construct_name = row['ConstructName'] if include_construct_name and 'ConstructName' in df.columns else None
                    subject_name = row['SubjectName'] if include_subject_name and 'SubjectName' in df.columns else None
                    
                    # Create the input text
                    input_text = self.create_input_text(
                        question, correct_answer, incorrect_answer, 
                        construct_name, subject_name, special_formatting
                    )
                    # Tokenize it
                    tokenized_data = self.tokenize_data(input_text)
                    
                    # Append the tokenized input ids and attention mask
                    inputs.append(tokenized_data['input_ids'])
                    labels.append(self.tokenize_data(misconception_name)['input_ids'])

                    if testing:
                        print(f"{input_text}")
                        print(f"Misconception: {misconception_name}\n\n")

        print(f"Sample input:\n\n{input_text} \n\nSample label: {misconception_name}")
        
        return {
            'instruction': torch.cat(inputs, dim=0),
            'chosen_response': torch.cat(labels, dim=0),
            'chosen_score': torch.ones(len(inputs)) 
        }

In [31]:
tokenizer_name = "Qwen/Qwen2.5-0.5B"
dataset_creator = DatasetCreator(tokenizer_name)

In [32]:
# Create the dataset
dpo_dataset = dataset_creator.create_dataset(
    train, misconception, 
    include_construct_name=True, include_subject_name=True, special_formatting=True, testing=False
)

Sample input:

 Here is a question for you:
Jo and Paul are arguing about how to fully describe the rotation from shape \( P \) to shape \( Q \)

Jo says: "a rotation of \( 90^{\degree} \) anticlockwise"

Paul says: "a rotation of \( +270^{\degree} \) about \( (-1,1)^{\prime \prime} \)

Who is correct? ![A coordinate grid with two trapeziums drawn on it and a centre of rotation marked on, one space to the right and one space up from the origin. Trapezium P has the coordinates: (-4,4) (-2,4) (-2,3) and (-4,1). Trapezium Q has the coordinates: (-4,-2) (-4,0) (-3,0) and (-1,-2).]()
The correct answer is: Only Paul
However, a common misconception is: Neither is correct
This question is related to: Describe a 90° or 270° rotation giving the angle and direction of rotation, and the coordinates of the centre of rotation, where the centre of rotation lies on the edge or outside of the object .
Subject area: Rotation. 

 Sample label: Does not know about the + notation for directions in rotatio

In [29]:
dpo_dataset['prompt'].shape, dpo_dataset['preferred_answer'].shape

(torch.Size([4370, 512]), torch.Size([4370, 512]))

In [16]:
train

,QuestionId,ConstructId,ConstructName,SubjectId,SubjectName,CorrectAnswer,QuestionText,AnswerAText,AnswerBText,AnswerCText,AnswerDText,MisconceptionAId,MisconceptionBId,MisconceptionCId,MisconceptionDId
0,0,856,Use the order of operations to carry out calcu...,33,BIDMAS,A,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times 2+(4-5) \),\( 3 \times(2+4-5) \),Does not need brackets,NaN,NaN,NaN,1672.0
1,1,1612,Simplify an algebraic fraction by factorising ...,1077,Simplifying Algebraic Fractions,D,"Simplify the following, if possible: \( \frac{...",\( m+1 \),\( m+2 \),\( m-1 \),Does not simplify,2142.0,143.0,2142.0,NaN
2,2,2774,Calculate the range from a list of data,339,Range and Interquartile Range from a List of Data,B,Tom and Katie are discussing the \( 5 \) plant...,Only\nTom,Only\nKatie,Both Tom and Katie,Neither is correct,1287.0,NaN,1287.0,1073.0
3,3,2377,Recall and use the intersecting diagonals prop...,88,Properties of Quadrilaterals,C,The angles highlighted on this rectangle with ...,acute,obtuse,\( 90^{\circ} \),Not enough information,1180.0,1180.0,NaN,1180.0
4,4,3387,Substitute positive integer values into formul...,67,Substitution into Formula,A,The equation \( f=3 r^{2}+3 \) is used to find...,\( 30 \),\( 27 \),\( 51 \),\( 24 \),NaN,NaN,NaN,1818.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1864,1864,2774,Calculate the range from a list of data,339,Range and Interquartile Range from a List of Data,C,What is the range of the following numbers?\n\...,\( 5 \),\( 11 \),\( 23 \),\( 16 \),2456.0,691.0,NaN,1349.0
1865,1865,2695,"Describe an enlargement, with no centre of enl...",90,Length Scale Factors in Similar Shapes,B,Shape \( Q \) is an enlargement of shape \( P ...,\( 3 \div 11 \),\( 11 \div 3 \),\( 3 \times 11 \),\( 11-3 \),1500.0,NaN,2442.0,1258.0
1866,1866,854,Use the order of operations to carry out calcu...,33,BIDMAS,B,What does the following equal?\n\[\n8-7+10 \ti...,\( 36 \),\( 31 \),\( -29 \),\( 33 \),NaN,NaN,2306.0,1507.0
1867,1867,2634,Distinguish between congruency and similarity,274,Congruency in Other Shapes,B,Tom and Katie are discussing congruence and si...,Only\nTom,Only Katie,Both Tom and Katie,Neither is correct,2312.0,NaN,2312.0,2312.0


In [17]:
misconception

,MisconceptionId,MisconceptionName
0,0,Does not know that angles in a triangle sum to...
1,1,Uses dividing fractions method for multiplying...
2,2,Believes there are 100 degrees in a full turn
3,3,Thinks a quadratic without a non variable term...
4,4,Believes addition of terms and powers of terms...
...,...,...
2582,2582,"When multiplying numbers with the same base, m..."
2583,2583,Does not know what a cube number is
2584,2584,Believes that any percentage of a larger numbe...
2585,2585,Believes a cubic expression should have three ...


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
import numpy as np
import random

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


/Users/seangorman/opt/anaconda3/envs/3dvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/seangorman/opt/anaconda3/envs/3dvenv/lib/python3.10/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


In [39]:
class DatasetCreator:
    def __init__(self, tokenizer_name, max_length=512, embedding_model_name='all-MiniLM-L6-v2'):
        """
        Initializes the DatasetCreator class.
        
        Parameters:
        - tokenizer_name: The name of the tokenizer (e.g., "gpt2", "bert-base-uncased")
        - max_length: Maximum length of the input sequence
        - embedding_model_name: The name of the embedding model for sentence transformers
        """
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.misconception_similarity_matrix = None

    def create_input_text(self, question, correct_answer, incorrect_answer, construct_name=None, subject_name=None, special_formatting=False):
        """
        Creates input text based on the question, correct answer, incorrect answer, construct name, and subject name.
        
        Parameters:
        - question: The question text
        - correct_answer: The correct answer text
        - incorrect_answer: The incorrect answer text
        - construct_name: The name of the construct (optional)
        - subject_name: The name of the subject (optional)
        - special_formatting: Whether to use special formatting for the prompt (optional)
        
        Returns:
        - input_text: Formatted text ready for tokenization
        """
        if special_formatting:
            input_text = (
                f"Here is a question for you:\n"
                f"{question}\n"
                f"The correct answer is: {correct_answer}\n"
                f"However, a common misconception is: {incorrect_answer}\n"
            )
            if construct_name:
                input_text += f"This question is related to: {construct_name}.\n"
            if subject_name:
                input_text += f"Subject area: {subject_name}."
        else:
            messages = [
                f"Question: {question}",
                f"Correct Answer: {correct_answer}",
                f"Incorrect Answer: {incorrect_answer}"
            ]
            
            if construct_name:
                messages.append(f"Construct Name: {construct_name}")
            if subject_name:
                messages.append(f"Subject Name: {subject_name}")
            
            input_text = "\n".join(messages)
        
        return input_text
    
    def tokenize_data(self, input_text):
        """
        Tokenizes the input text using the provided tokenizer.
        
        Parameters:
        - input_text: The input text to be tokenized
        
        Returns:
        - tokenized_data: Tokenized representation of the input text
        """
        return self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
    
    def create_similarity_matrix(self, misconception_mapping, embedding_model_name=None):
        """
        Creates a similarity matrix of all the misconception names using sentence embeddings.
        
        Parameters:
        - misconception_mapping: A DataFrame containing the Misconception IDs and their names
        - embedding_model_name: The name of the embedding model (optional)
        
        Returns:
        - similarity_matrix: A matrix representing similarity between all misconception names
        """
        if embedding_model_name:
            self.embedding_model = SentenceTransformer(embedding_model_name)
        
        misconception_names = misconception_mapping['MisconceptionName'].tolist()
        embeddings = self.embedding_model.encode(misconception_names, convert_to_tensor=True, show_progress_bar=True)
        #similarity_matrix = torch.nn.functional.cosine_similarity(embeddings.unsqueeze(1), embeddings.unsqueeze(0), dim=-1)
        similarity_matrix = torch.nn.functional.cosine_similarity(embeddings.unsqueeze(1).cpu(), embeddings.unsqueeze(0).cpu(), dim=-1)
        self.misconception_similarity_matrix = similarity_matrix
        
        return similarity_matrix
    
    def create_dataset(self, df, misconception_mapping, include_construct_name=False, include_subject_name=False, special_formatting=False, scoring_type='binary', return_type='tokenized', testing=False):
        """
        Creates a dataset for direct preference optimization (DPO).
        
        Parameters:
        - df: The training DataFrame containing all the necessary columns
        - misconception_mapping: A DataFrame containing the Misconception IDs and their names
        - include_construct_name: Whether to include the construct name in the prompt (optional)
        - include_subject_name: Whether to include the subject name in the prompt (optional)
        - special_formatting: Whether to use special formatting for the prompts (optional)
        - scoring_type: The type of scoring ('binary' or 'graded')
        - return_type: The type of return ('tokenized' or 'text')
        
        Returns:
        - A dictionary containing either tokenized input tensors or text for the DPO model
        """
        inputs, labels, rejected_responses, chosen_scores, rejected_scores = [], [], [], [], []
        misconception_dict = pd.Series(misconception_mapping.MisconceptionName.values, index=misconception_mapping.MisconceptionId).to_dict()
        
        if testing:
            df = df.head(3)
        
        for index, row in df.iterrows():
            question = row['QuestionText']
            correct_answer = row[f'Answer{row.CorrectAnswer}Text']
            
            for answer_label in ['A', 'B', 'C', 'D']:
                misconception_id = row.get(f'Misconception{answer_label}Id', None)
                if pd.notna(misconception_id):
                    incorrect_answer = row[f'Answer{answer_label}Text']
                    misconception_name = misconception_dict.get(misconception_id, "Unknown Misconception")
                    
                    construct_name = row['ConstructName'] if include_construct_name and 'ConstructName' in df.columns else None
                    subject_name = row['SubjectName'] if include_subject_name and 'SubjectName' in df.columns else None
                    
                    # Create the input text
                    input_text = self.create_input_text(
                        question, correct_answer, incorrect_answer, 
                        construct_name, subject_name, special_formatting
                    )
                    
                    # Determine scoring
                    if scoring_type == 'binary':
                        least_similar_idx = torch.argmin(self.misconception_similarity_matrix[int(misconception_id)]).item()
                        chosen_score = 1
                        rejected_score = 0
                        rejected_response = misconception_dict[least_similar_idx]
                    elif scoring_type == 'graded':
                        chosen_score = 5
                        random_idx = random.choice(range(len(self.misconception_similarity_matrix)))
                        similarity = self.misconception_similarity_matrix[int(misconception_id), int(random_idx)].item()
                        rejected_score = max(0, similarity * 5)  # Sharpened similarity scaled by 5
                        rejected_response = misconception_dict[random_idx]
                    else:
                        raise ValueError("Invalid scoring type. Choose either 'binary' or 'graded'.")
                    
                    chosen_scores.append(chosen_score)
                    rejected_scores.append(rejected_score)
                    
                    if return_type == 'tokenized':
                        tokenized_data = self.tokenize_data(input_text)
                        inputs.append(tokenized_data['input_ids'])
                        labels.append(self.tokenize_data(misconception_name)['input_ids'])
                        rejected_responses.append(self.tokenize_data(rejected_response)['input_ids'])
                    elif return_type == 'text':
                        inputs.append(input_text)
                        labels.append(misconception_name)
                        rejected_responses.append(rejected_response)
                    else:
                        raise ValueError("Invalid return type. Choose either 'tokenized' or 'text'.")

                    if testing:
                        print(f"{input_text}")
                        print(f"chosen_response: {misconception_name} Chosen Score: {chosen_score}, Rejected_response: {rejected_response} ,Rejected Score: {rejected_score}\n\n")
        
        if return_type == 'tokenized':
            return {
                'instruction': torch.cat(inputs, dim=0),
                'chosen_response': torch.cat(labels, dim=0),
                'rejected_response': torch.cat(rejected_responses, dim=0),
                'chosen_score': torch.tensor(chosen_scores, dtype=torch.float32),
                'rejected_score': torch.tensor(rejected_scores, dtype=torch.float32)
            }
        elif return_type == 'text':
            return {
                'instruction': inputs,
                'chosen_response': labels,
                'rejected_response': rejected_responses,
                'chosen_score': chosen_scores,
                'rejected_score': rejected_scores
            }

In [40]:
tokenizer_name = "Qwen/Qwen2.5-0.5B"
embedding_model_name = 'all-MiniLM-L6-v2'
dataset_creator = DatasetCreator(tokenizer_name, embedding_model_name=embedding_model_name)


In [41]:
#sim_matric = dataset_creator.create_similarity_matrix(misconception)
dataset_creator.misconception_similarity_matrix = similarity_matrix

In [43]:
# Create the dataset
dpo_dataset = dataset_creator.create_dataset(
    train, misconception, 
    include_construct_name=True, include_subject_name=True, special_formatting=True, scoring_type='binary', return_type='text', testing=True
)


Here is a question for you:
\[
3 \times 2+4-5
\]
Where do the brackets need to go to make the answer equal \( 13 \) ?
The correct answer is: \( 3 \times(2+4)-5 \)
However, a common misconception is: Does not need brackets
This question is related to: Use the order of operations to carry out calculations involving powers.
Subject area: BIDMAS.
chosen_response: Confuses the order of operations, believes addition comes before multiplication  Chosen Score: 1, Rejected_response: Estimates obtuse angles as 90 degrees or less ,Rejected Score: 0


Here is a question for you:
Simplify the following, if possible: \( \frac{m^{2}+2 m-3}{m-3} \)
The correct answer is: Does not simplify
However, a common misconception is: \( m+1 \)
This question is related to: Simplify an algebraic fraction by factorising the numerator.
Subject area: Simplifying Algebraic Fractions.
chosen_response: Does not know that to factorise a quadratic expression, to find two numbers that add to give the coefficient of the x 

In [44]:
dpo_dataset

{'instruction': ['Here is a question for you:\n\\[\n3 \\times 2+4-5\n\\]\nWhere do the brackets need to go to make the answer equal \\( 13 \\) ?\nThe correct answer is: \\( 3 \\times(2+4)-5 \\)\nHowever, a common misconception is: Does not need brackets\nThis question is related to: Use the order of operations to carry out calculations involving powers.\nSubject area: BIDMAS.',
  'Here is a question for you:\nSimplify the following, if possible: \\( \\frac{m^{2}+2 m-3}{m-3} \\)\nThe correct answer is: Does not simplify\nHowever, a common misconception is: \\( m+1 \\)\nThis question is related to: Simplify an algebraic fraction by factorising the numerator.\nSubject area: Simplifying Algebraic Fractions.',
  'Here is a question for you:\nSimplify the following, if possible: \\( \\frac{m^{2}+2 m-3}{m-3} \\)\nThe correct answer is: Does not simplify\nHowever, a common misconception is: \\( m+2 \\)\nThis question is related to: Simplify an algebraic fraction by factorising the numerator.\n